# Overtime-Only Production Scheduling Model

This notebook is the second step of the planning workflow:

1. The regular-time notebook writes completed units to `Product_Forecast.Solved_Regular`.
2. Excel or Python calculates `For_OT = Forecast_Amount - Solved_Regular`.
3. This notebook treats `For_OT` as its demand limit and schedules only registered overtime capacity.

This first OT version is intentionally conservative. Every `For_OT` unit must pass through its complete route using OT production only, starting with zero OT WIP. It does not reuse `Last_MIP` or the regular solver's intermediate-stage output, which prevents double-counting inventory.

## Terminology

- **Process** is a production method or stage such as `Pri`, `Cut`, `Sew`, or `Pac`.
- **Line** is a physical resource inside a process, such as `Pri1` or `Pri2`.
- **POL** remains indexed by `(product, process)`.
- **OT production and capacity** remain indexed by physical line and date.


In [ ]:
from pathlib import Path
import importlib
import re

from IPython.display import display
import pandas as pd
import pulp as pl

from utils import util_display as display_utils

# Reload the local display helper while developing in Jupyter.
importlib.reload(display_utils)


## 1. Settings and file paths


In [ ]:
MS_PER_HOUR = 3_600_000

DAY_SHIFT_NAME = "Day"
MIN_OT_TIME_UTILIZATION = 0.70
OT_BLOCK_TIME_MS = 30 * 60 * 1_000
OT_PAY_MULTIPLIER = 1.5

ALLOW_SAME_DAY_TRANSFER = True
CLEAR_ENDING_OT_WIP = False

SOLVER_RELATIVE_GAP = 0.01
SOLVER_TIME_LIMIT_SECONDS = 1800
SOLVER_LOG = True
SOLUTION_TOLERANCE = 1e-6

OT_TOTAL_HEADER_ROW = 2
OT_TOTAL_DATE_START_COLUMN = 9

DATA_DIR = Path("xlconfigs")
PRODUCTION_INPUT = DATA_DIR / "production_input.xlsm"
LABOR_INPUT = DATA_DIR / "labor_input.xlsm"

for input_file in (PRODUCTION_INPUT, LABOR_INPUT):
    if not input_file.exists():
        raise FileNotFoundError(
            f"Input file not found: {input_file.resolve()}"
        )


## 2. Input-validation helpers


In [ ]:
def require_columns(frame, required_columns, sheet_name):
    missing_columns = [
        column
        for column in required_columns
        if column not in frame.columns
    ]
    if missing_columns:
        raise ValueError(
            f"{sheet_name} is missing columns: {missing_columns}"
        )


def clean_identifier(value):
    return str(value).strip()


def clean_identifier_list(series):
    return [
        clean_identifier(value)
        for value in series.dropna()
    ]


def require_unique(values, label):
    values = pd.Series(values, dtype="object")
    duplicates = values[values.duplicated()].unique().tolist()
    if duplicates:
        raise ValueError(f"Duplicate {label}: {duplicates}")


def as_integers(series, label):
    numeric = pd.to_numeric(series, errors="raise")
    if numeric.isna().any():
        raise ValueError(f"{label} contains blank values.")

    fractional = (numeric - numeric.round()).abs() > 1e-6
    if fractional.any():
        raise ValueError(f"{label} must contain whole units.")

    return numeric.round().astype("int64")


def as_nonnegative_integers(series, label):
    integers = as_integers(series, label)
    if (integers < 0).any():
        raise ValueError(f"{label} cannot contain negative values.")
    return integers


def process_from_line(line):
    return re.sub(r"\d+$", "", clean_identifier(line)).strip()


def safe_ratio(numerator, denominator):
    return numerator / denominator if denominator > 0 else 0.0


## 3. Load products, OT demand, lines, and process routes


In [ ]:
production_sheet_names = [
    "Master_List",
    "Product_Forecast",
    "POL_Matrix",
    "Line_Config",
]

with pd.ExcelFile(PRODUCTION_INPUT, engine="openpyxl") as workbook:
    missing_sheets = [
        sheet
        for sheet in production_sheet_names
        if sheet not in workbook.sheet_names
    ]
    if missing_sheets:
        raise ValueError(
            f"{PRODUCTION_INPUT.name} is missing sheets: {missing_sheets}"
        )

    production_sheets = {
        sheet: workbook.parse(sheet)
        for sheet in production_sheet_names
    }

master_list = production_sheets["Master_List"]
product_forecast = production_sheets["Product_Forecast"]
pol_matrix_raw = production_sheets["POL_Matrix"]
line_config = production_sheets["Line_Config"]


In [ ]:
require_columns(
    master_list,
    ["Product_List", "Production_Lines"],
    "Master_List",
)

products = clean_identifier_list(master_list["Product_List"])
lines = clean_identifier_list(master_list["Production_Lines"])

require_unique(products, "products in Master_List")
require_unique(lines, "lines in Master_List")

line_process = {
    line: process_from_line(line)
    for line in lines
}


forecast_columns = [
    "Product_List",
    "Forecast_Amount",
    "Solved_Regular",
    "For_OT",
    "Profit_Per_Product",
]
require_columns(product_forecast, forecast_columns, "Product_Forecast")

product_forecast = (
    product_forecast.loc[
        product_forecast["Product_List"].notna(),
        forecast_columns,
    ]
    .assign(
        Product_List=lambda df: df["Product_List"].map(
            clean_identifier
        )
    )
)
require_unique(
    product_forecast["Product_List"],
    "products in Product_Forecast",
)

unknown_products = sorted(
    set(product_forecast["Product_List"]) - set(products)
)
missing_products = sorted(
    set(products) - set(product_forecast["Product_List"])
)
if unknown_products or missing_products:
    raise ValueError(
        "Product_Forecast does not match Master_List. "
        f"Unknown={unknown_products}, missing={missing_products}"
    )

product_forecast = product_forecast.set_index(
    "Product_List"
).reindex(products)

for column in ["Forecast_Amount", "Solved_Regular", "For_OT"]:
    product_forecast[column] = as_nonnegative_integers(
        product_forecast[column],
        f"Product_Forecast.{column}",
    )

product_forecast["Profit_Per_Product"] = pd.to_numeric(
    product_forecast["Profit_Per_Product"],
    errors="raise",
)

expected_for_ot = (
    product_forecast["Forecast_Amount"]
    - product_forecast["Solved_Regular"]
)
invalid_handoff = (
    expected_for_ot.lt(0)
    | product_forecast["For_OT"].ne(expected_for_ot)
)
if invalid_handoff.any():
    bad_products = product_forecast.index[invalid_handoff].tolist()
    raise ValueError(
        "For_OT must equal Forecast_Amount - Solved_Regular. "
        f"Check products: {bad_products}"
    )

ot_demand = product_forecast["For_OT"].to_dict()
profit_per_product = product_forecast[
    "Profit_Per_Product"
].to_dict()


line_config_columns = ["Production_Line", "Minimum_Input"]
require_columns(line_config, line_config_columns, "Line_Config")

line_config = line_config.loc[
    line_config["Production_Line"].notna(),
    line_config_columns,
].copy()
line_config["Production_Line"] = line_config[
    "Production_Line"
].map(clean_identifier)
require_unique(
    line_config["Production_Line"],
    "lines in Line_Config",
)

if set(line_config["Production_Line"]) != set(lines):
    raise ValueError("Line_Config lines do not match Master_List.")

line_config = line_config.set_index(
    "Production_Line"
).reindex(lines)
line_config["Minimum_Input"] = as_nonnegative_integers(
    line_config["Minimum_Input"],
    "Line_Config.Minimum_Input",
)
minimum_input_by_line = line_config["Minimum_Input"].to_dict()


In [ ]:
pol_matrix_raw.columns = [
    clean_identifier(column)
    for column in pol_matrix_raw.columns
]
require_columns(pol_matrix_raw, ["Product_List"], "POL_Matrix")

pol_matrix_raw = pol_matrix_raw.loc[
    pol_matrix_raw["Product_List"].notna()
].copy()
pol_matrix_raw["Product_List"] = pol_matrix_raw[
    "Product_List"
].map(clean_identifier)
require_unique(
    pol_matrix_raw["Product_List"],
    "products in POL_Matrix",
)

if set(pol_matrix_raw["Product_List"]) != set(products):
    raise ValueError("POL_Matrix products do not match Master_List.")

pol_matrix = (
    pol_matrix_raw.set_index("Product_List")
    .reindex(products)
    .apply(pd.to_numeric, errors="raise")
)

if pol_matrix.isna().any().any():
    raise ValueError("POL_Matrix contains blank labor-time cells.")
if (pol_matrix < 0).any().any():
    raise ValueError("POL_Matrix labor times cannot be negative.")
if ((pol_matrix - pol_matrix.round()).abs() > 1e-6).any().any():
    raise ValueError("POL_Matrix labor times must be whole milliseconds.")

pol_matrix = pol_matrix.round().astype("int64")
registered_processes = list(dict.fromkeys(line_process.values()))
process_order = [
    process
    for process in pol_matrix.columns
    if process in registered_processes
]

unmapped_processes = [
    process
    for process in pol_matrix.columns
    if (pol_matrix[process] > 0).any()
    and process not in registered_processes
]
if unmapped_processes:
    raise ValueError(
        "Required processes have no registered line: "
        f"{unmapped_processes}"
    )

product_routes = {
    product: [
        process
        for process in process_order
        if pol_matrix.loc[product, process] > 0
    ]
    for product in products
}

products_without_routes = [
    product
    for product, route in product_routes.items()
    if not route
]
if products_without_routes:
    raise ValueError(
        f"Products without a route: {products_without_routes}"
    )

process_lines = {
    process: [
        line
        for line in lines
        if line_process[line] == process
    ]
    for process in process_order
}

labor_time_per_unit_ms = {
    (product, process): int(pol_matrix.loc[product, process])
    for product in products
    for process in product_routes[product]
}


## 4. Load registered overtime and its calendar

`OT_Total` remains in hours in Excel. Python converts it to milliseconds immediately. Only staffed `Day` lines are eligible.

Worker decisions use 30-minute blocks. A line-day receives variables only for blocks inside its registered OT duration, so workers cannot be moved into later unregistered time. The 70% minimum applies to the worker-block capacity actually scheduled, while `OT_Total` remains the maximum availability.


In [ ]:
with pd.ExcelFile(LABOR_INPUT, engine="openpyxl") as workbook:
    required_sheets = {"Expanded_Details", "OT_Total"}
    missing_sheets = sorted(
        required_sheets - set(workbook.sheet_names)
    )
    if missing_sheets:
        raise ValueError(
            f"{LABOR_INPUT.name} is missing sheets: {missing_sheets}"
        )

    shift_details = workbook.parse("Expanded_Details")
    ot_total_raw = workbook.parse(
        "OT_Total",
        header=OT_TOTAL_HEADER_ROW,
    )

shift_columns = [
    "Production_Line",
    "Shift",
    "Workers",
    "Max_Hr",
    "Hrly_Sal",
]
require_columns(shift_details, shift_columns, "Expanded_Details")

shift_details = shift_details.loc[
    shift_details["Production_Line"].notna()
    & shift_details["Shift"].notna(),
    shift_columns,
].copy()
shift_details["Production_Line"] = shift_details[
    "Production_Line"
].map(clean_identifier)
shift_details["Shift"] = shift_details["Shift"].map(
    clean_identifier
)

if shift_details.duplicated(
    subset=["Production_Line", "Shift"]
).any():
    raise ValueError("Expanded_Details has duplicate line-shift rows.")

shift_details["Workers"] = as_nonnegative_integers(
    shift_details["Workers"],
    "Expanded_Details.Workers",
)
shift_details[["Max_Hr", "Hrly_Sal"]] = shift_details[
    ["Max_Hr", "Hrly_Sal"]
].apply(pd.to_numeric, errors="raise")

day_shift_details = shift_details.loc[
    shift_details["Shift"].eq(DAY_SHIFT_NAME)
    & shift_details["Workers"].gt(0)
].copy()

if (day_shift_details["Max_Hr"] <= 0).any():
    raise ValueError("Active Day lines need positive Max_Hr.")
if (day_shift_details["Hrly_Sal"] < 0).any():
    raise ValueError("Hrly_Sal cannot be negative.")

day_shift_details = day_shift_details.assign(
    Max_Time_MS=lambda df: (
        df["Max_Hr"] * MS_PER_HOUR
    ).round().astype("int64"),
    Salary_Per_MS=lambda df: (
        df["Hrly_Sal"] / df["Max_Time_MS"]
    ),
)

day_shift_lines = [
    line
    for line in lines
    if line in set(day_shift_details["Production_Line"])
]
line_workers = day_shift_details.set_index(
    "Production_Line"
)["Workers"].to_dict()
salary_per_ms = day_shift_details.set_index(
    "Production_Line"
)["Salary_Per_MS"].to_dict()


In [ ]:
if len(ot_total_raw.columns) <= OT_TOTAL_DATE_START_COLUMN:
    raise ValueError("OT_Total has no dated availability columns.")

calendar_dates = [
    pd.Timestamp(pd.to_datetime(column, errors="raise")).normalize()
    for column in ot_total_raw.columns[OT_TOTAL_DATE_START_COLUMN:]
]
require_unique(calendar_dates, "dates in OT_Total")
if calendar_dates != sorted(calendar_dates):
    raise ValueError("OT_Total dates must be chronological.")

days = list(range(1, len(calendar_dates) + 1))
day_to_date = dict(zip(days, calendar_dates))
date_to_day = dict(zip(calendar_dates, days))

ot_total = ot_total_raw.iloc[
    :,
    [
        0,
        1,
        2,
        *range(
            OT_TOTAL_DATE_START_COLUMN,
            len(ot_total_raw.columns),
        ),
    ],
].copy()
ot_total.columns = [
    "Line",
    "Shift",
    "Workers",
    *calendar_dates,
]

ot_total["Line"] = ot_total["Line"].ffill()
ot_total = ot_total.loc[
    ot_total["Shift"].notna()
    & ot_total["Line"].ne("Grand Total")
].copy()
ot_total["Line"] = ot_total["Line"].map(clean_identifier)
ot_total["Shift"] = ot_total["Shift"].map(clean_identifier)

unknown_ot_lines = sorted(set(ot_total["Line"]) - set(lines))
if unknown_ot_lines:
    raise ValueError(
        f"OT_Total contains unregistered lines: {unknown_ot_lines}"
    )

non_day_shifts = sorted(set(ot_total["Shift"]) - {DAY_SHIFT_NAME})
if non_day_shifts:
    raise ValueError(
        f"OT_Total supports only {DAY_SHIFT_NAME!r}; "
        f"found {non_day_shifts}"
    )

ot_long = (
    ot_total.melt(
        id_vars=["Line", "Shift"],
        value_vars=calendar_dates,
        var_name="Date",
        value_name="OT_Time_MS",
    )
    .assign(
        Day=lambda df: df["Date"].map(date_to_day),
        OT_Time_MS=lambda df: (
            pd.to_numeric(df["OT_Time_MS"], errors="raise")
            .fillna(0.0)
            .mul(MS_PER_HOUR)
            .round()
            .astype("int64")
        ),
    )
)

if (ot_long["OT_Time_MS"] < 0).any():
    raise ValueError("OT_Total cannot contain negative time.")
if ot_long.duplicated(subset=["Line", "Shift", "Day"]).any():
    raise ValueError("OT_Total contains duplicate line-shift-day rows.")

excel_ot_time_ms = ot_long.set_index(
    ["Line", "Shift", "Day"]
)["OT_Time_MS"].to_dict()

ot_available_time_ms = {}
ot_available_labor_ms = {}
ot_labor_capacity_ms = {}
ot_blocks_by_line_day = {}

for line in day_shift_lines:
    workers = int(line_workers[line])

    for day in days:
        key = (line, day)
        available_time_ms = int(
            excel_ot_time_ms.get(
                (line, DAY_SHIFT_NAME, day),
                0,
            )
        )
        time_blocks = available_time_ms // OT_BLOCK_TIME_MS

        ot_available_time_ms[key] = available_time_ms
        ot_blocks_by_line_day[key] = list(
            range(1, time_blocks + 1)
        )
        ot_available_labor_ms[key] = available_time_ms * workers
        ot_labor_capacity_ms[key] = (
            time_blocks * workers * OT_BLOCK_TIME_MS
        )

overtime_line_day_keys = [
    (line, day)
    for line in day_shift_lines
    for day in days
    if ot_labor_capacity_ms[line, day] > 0
]
overtime_line_day_set = set(overtime_line_day_keys)

active_ot_lines = [
    line
    for line in day_shift_lines
    if any(
        active_line == line
        for active_line, day in overtime_line_day_keys
    )
]
spare_ot_lines = [
    line
    for line in day_shift_lines
    if line not in active_ot_lines
]

overtime_worker_keys = [
    (line, day, block)
    for line, day in overtime_line_day_keys
    for block in ot_blocks_by_line_day[line, day]
]

allowed_ot_workers = {
    (line, day, block): int(line_workers[line])
    for line, day, block in overtime_worker_keys
}

ot_pay_per_worker_block = {
    line: (
        salary_per_ms[line]
        * OT_BLOCK_TIME_MS
        * OT_PAY_MULTIPLIER
    )
    for line in active_ot_lines
}


## 5. Feasible OT index sets and decision variables


In [ ]:
active_process_lines = {
    process: [
        line
        for line in process_lines[process]
        if line in active_ot_lines
    ]
    for process in process_order
}

unavailable_required_processes = [
    process
    for process in process_order
    if any(
        ot_demand[product] > 0
        and process in product_routes[product]
        for product in products
    )
    and not active_process_lines[process]
]
if unavailable_required_processes:
    print(
        "Warning: required processes have no registered OT line: "
        f"{unavailable_required_processes}"
    )

feasible_product_line_pairs = [
    (product, line)
    for product in products
    for line in active_ot_lines
    if line_process[line] in product_routes[product]
]

products_by_line = {
    line: [
        product
        for product, feasible_line in feasible_product_line_pairs
        if feasible_line == line
    ]
    for line in active_ot_lines
}

overtime_production_keys = [
    (product, line, day)
    for product, line in feasible_product_line_pairs
    for day in days
    if (line, day) in overtime_line_day_set
]

wip_keys = [
    (product, process, day)
    for product in products
    for process in product_routes[product][:-1]
    for day in days
]


In [ ]:
model = pl.LpProblem(
    "Overtime_Production_Scheduling",
    pl.LpMaximize,
)

production = pl.LpVariable.dicts(
    "OTFinishedProduction",
    [(product, day) for product in products for day in days],
    lowBound=0,
    cat="Integer",
)

overtime_qty = pl.LpVariable.dicts(
    "OvertimeQty",
    overtime_production_keys,
    lowBound=0,
    cat="Integer",
)

batch_active = pl.LpVariable.dicts(
    "OTBatchActive",
    overtime_production_keys,
    cat="Binary",
)

overtime_workers = pl.LpVariable.dicts(
    "OvertimeWorkers",
    overtime_worker_keys,
    lowBound=0,
    cat="Integer",
)

wip = pl.LpVariable.dicts(
    "OTWIP",
    wip_keys,
    lowBound=0,
    cat="Integer",
)


## 6. OT production, route flow, and WIP expressions


In [ ]:
stage_processed = {
    (product, process, day): pl.lpSum(
        overtime_qty[product, line, day]
        for line in active_process_lines[process]
        if (product, line, day) in overtime_qty
    )
    for product in products
    for process in product_routes[product]
    for day in days
}

ot_day_processed = {
    (line, day): pl.lpSum(
        overtime_qty[product, line, day]
        for product in products_by_line[line]
        if (product, line, day) in overtime_qty
    )
    for line, day in overtime_line_day_keys
}

overtime_labor_used_ms = {
    (line, day): pl.lpSum(
        labor_time_per_unit_ms[product, line_process[line]]
        * overtime_qty[product, line, day]
        for product in products_by_line[line]
        if (product, line, day) in overtime_qty
    )
    for line, day in overtime_line_day_keys
}


## 7. Remaining-demand and route constraints

`For_OT` replaces the regular notebook's forecast limit. Opening OT WIP is zero. With same-day transfer enabled, upstream and downstream OT processes may work on the same units on the same date.


In [ ]:
for product in products:
    route = product_routes[product]
    first_process = route[0]
    final_process = route[-1]

    model += (
        pl.lpSum(production[product, day] for day in days)
        <= ot_demand[product],
        f"OT_Demand_Limit_{product}",
    )
    model += (
        pl.lpSum(
            stage_processed[product, first_process, day]
            for day in days
        )
        <= ot_demand[product],
        f"OT_Route_Start_Limit_{product}",
    )

    for day in days:
        model += (
            production[product, day]
            == stage_processed[product, final_process, day],
            f"OT_Finished_Output_{product}_{day}",
        )

    for current_process, next_process in zip(route, route[1:]):
        for day_position, day in enumerate(days):
            if day_position == 0:
                opening_units = 0
            else:
                previous_day = days[day_position - 1]
                opening_units = wip[
                    product,
                    current_process,
                    previous_day,
                ]

            model += (
                wip[product, current_process, day]
                == opening_units
                + stage_processed[product, current_process, day]
                - stage_processed[product, next_process, day],
                f"OT_WIP_Balance_{product}_{current_process}_{day}",
            )

            if not ALLOW_SAME_DAY_TRANSFER:
                model += (
                    stage_processed[product, next_process, day]
                    <= opening_units,
                    f"OT_Prior_Day_Release_{product}_{next_process}_{day}",
                )

        if CLEAR_ENDING_OT_WIP:
            model += (
                wip[product, current_process, days[-1]] == 0,
                f"Clear_Ending_OT_WIP_{product}_{current_process}",
            )


## 8. Minimum batches and OT labor capacity


In [ ]:
for product, line, day in overtime_production_keys:
    quantity = overtime_qty[product, line, day]
    active = batch_active[product, line, day]

    model += (
        quantity >= minimum_input_by_line[line] * active,
        f"OT_Minimum_Input_{product}_{line}_{day}",
    )
    model += (
        quantity <= ot_demand[product] * active,
        f"OT_Batch_Activation_{product}_{line}_{day}",
    )


for line, day in overtime_line_day_keys:
    key = (line, day)
    used_labor_ms = overtime_labor_used_ms[key]

    for block in ot_blocks_by_line_day[key]:
        model += (
            overtime_workers[line, day, block]
            <= allowed_ot_workers[line, day, block],
            f"OT_Worker_Limit_{line}_{day}_{block}",
        )

    for current_block, next_block in zip(
        ot_blocks_by_line_day[key],
        ot_blocks_by_line_day[key][1:],
    ):
        model += (
            overtime_workers[line, day, next_block]
            <= overtime_workers[line, day, current_block],
            f"OT_Continuation_{line}_{day}_{next_block}",
        )

    total_worker_blocks = pl.lpSum(
        overtime_workers[line, day, block]
        for block in ot_blocks_by_line_day[key]
    )
    scheduled_labor_ms = OT_BLOCK_TIME_MS * total_worker_blocks

    model += (
        used_labor_ms <= scheduled_labor_ms,
        f"OT_Time_Max_{line}_{day}",
    )
    model += (
        used_labor_ms
        >= MIN_OT_TIME_UTILIZATION * scheduled_labor_ms,
        f"OT_Time_Min_{line}_{day}",
    )


## 9. Objective and solve


In [ ]:
total_ot_production_profit = pl.lpSum(
    profit_per_product[product] * production[product, day]
    for product in products
    for day in days
)

total_ot_labor_cost = pl.lpSum(
    ot_pay_per_worker_block[line]
    * overtime_workers[line, day, block]
    for line, day, block in overtime_worker_keys
)

model += (
    total_ot_production_profit - total_ot_labor_cost,
    "Maximize_OT_Net_Profit",
)

num_binary = sum(
    variable.isBinary()
    for variable in model.variables()
)
num_integer = sum(
    variable.cat == pl.LpInteger and not variable.isBinary()
    for variable in model.variables()
)
num_continuous = sum(
    variable.cat == pl.LpContinuous
    for variable in model.variables()
)

print(f"Total Variables:   {len(model.variables()):,}")
print(f"  • Continuous:    {num_continuous:,}")
print(f"  • General Int:   {num_integer:,}")
print(f"  • Binary (0/1):  {num_binary:,}")
print(f"Total Constraints: {len(model.constraints):,}")

solver = pl.HiGHS(
    gapRel=SOLVER_RELATIVE_GAP,
    timeLimit=SOLVER_TIME_LIMIT_SECONDS,
    msg=SOLVER_LOG,
)
model.solve(solver)

solve_status = pl.LpStatus[model.status]
print("Status:", solve_status)

if solve_status not in {"Optimal", "Feasible"}:
    raise RuntimeError(
        f"The OT model has no reportable solution: {solve_status}"
    )


# Reports


In [ ]:
def solution_value(item):
    value = pl.value(item)
    return 0.0 if value is None else float(value)


objective_summary = pd.Series({
    "OT Finished-Production Profit": solution_value(
        total_ot_production_profit
    ),
    "OT Labor Cost": solution_value(total_ot_labor_cost),
    "OT Net Objective": solution_value(model.objective),
})

print("OT OBJECTIVE SUMMARY")
display(objective_summary.to_frame("Value").style.format("{:,.2f}"))


## 10. OT demand attainment


In [ ]:
attainment_rows = []

for product in products:
    solved_ot = sum(
        solution_value(production[product, day])
        for day in days
    )
    attainment_rows.append({
        "Product": product,
        "Original Forecast": product_forecast.loc[
            product, "Forecast_Amount"
        ],
        "Solved Regular": product_forecast.loc[
            product, "Solved_Regular"
        ],
        "For OT": ot_demand[product],
        "Solved OT": solved_ot,
        "Remaining After OT": max(0.0, ot_demand[product] - solved_ot),
        "OT Attainment": safe_ratio(solved_ot, ot_demand[product]),
    })

ot_attainment_table = pd.DataFrame(
    attainment_rows
).set_index("Product")

print("OT DEMAND ATTAINMENT")
display(
    ot_attainment_table.style.format({
        "Original Forecast": "{:,.0f}",
        "Solved Regular": "{:,.0f}",
        "For OT": "{:,.0f}",
        "Solved OT": "{:,.0f}",
        "Remaining After OT": "{:,.0f}",
        "OT Attainment": "{:.1%}",
    })
)


## 11. Finished OT output and line schedule


In [ ]:
schedule_dates = [
    pd.Timestamp(day_to_date[day]).date()
    for day in days
]

finished_ot_table = pd.DataFrame(
    {
        pd.Timestamp(day_to_date[day]).date(): [
            round(solution_value(production[product, day]))
            for product in products
        ]
        for day in days
    },
    index=pd.Index(products, name="Product"),
)
finished_ot_table["Total"] = finished_ot_table.sum(axis=1)

print("FINISHED OT OUTPUT")
display(finished_ot_table.style.format("{:,.0f}"))


overtime_rows = [
    {
        "Date": pd.Timestamp(day_to_date[day]).date(),
        "Line": line,
        "Product": product,
        "Quantity": solution_value(
            overtime_qty[product, line, day]
        ),
    }
    for product, line, day in overtime_production_keys
    if solution_value(
        overtime_qty[product, line, day]
    ) > SOLUTION_TOLERANCE
]

if overtime_rows:
    overtime_schedule_table = (
        pd.DataFrame(overtime_rows)
        .pivot_table(
            index=["Line", "Product"],
            columns="Date",
            values="Quantity",
            aggfunc="sum",
            fill_value=0,
            sort=False,
        )
        .reindex(columns=schedule_dates, fill_value=0)
        .round()
        .astype(int)
    )
    overtime_schedule_table["Total"] = (
        overtime_schedule_table.sum(axis=1)
    )
    overtime_schedule_table.columns.name = None

    print("OVERTIME PRODUCTION SCHEDULE")
    display(
        display_utils.style_grouped_table(
            overtime_schedule_table,
            {
                column: "{:,.0f}"
                for column in overtime_schedule_table.columns
            },
        )
    )
else:
    overtime_schedule_table = pd.DataFrame()
    print("No overtime production was scheduled.")


## 12. OT labor requirement and cost


In [ ]:
ot_labor_rows = []

for line, day in overtime_line_day_keys:
    key = (line, day)
    used_labor_ms = solution_value(overtime_labor_used_ms[key])
    if used_labor_ms <= SOLUTION_TOLERANCE:
        continue

    scheduled_worker_time_ms = OT_BLOCK_TIME_MS * sum(
        solution_value(overtime_workers[line, day, block])
        for block in ot_blocks_by_line_day[key]
    )
    line_day_cost = ot_pay_per_worker_block[line] * sum(
        solution_value(overtime_workers[line, day, block])
        for block in ot_blocks_by_line_day[key]
    )

    ot_labor_rows.append({
        "Date": pd.Timestamp(day_to_date[day]).date(),
        "Process": line_process[line],
        "Line": line,
        "OT Units": solution_value(ot_day_processed[key]),
        "Minimum Time Utilization": MIN_OT_TIME_UTILIZATION,
        "Actual Time Utilization": safe_ratio(
            used_labor_ms,
            scheduled_worker_time_ms,
        ),
        "Available OT Hours per Worker": (
            ot_available_time_ms[key] / MS_PER_HOUR
        ),
        "Registered OT Worker-Hours": (
            ot_available_labor_ms[key] / MS_PER_HOUR
        ),
        "Used OT Worker-Hours": used_labor_ms / MS_PER_HOUR,
        "Scheduled OT Worker-Hours": (
            scheduled_worker_time_ms / MS_PER_HOUR
        ),
        "Line Workers": line_workers[line],
        "OT Labor Cost": line_day_cost,
    })

if ot_labor_rows:
    ot_labor_table = pd.DataFrame(ot_labor_rows).set_index(
        ["Date", "Process", "Line"]
    )
    print("OVERTIME LABOR REQUIREMENT")
    display(
        display_utils.style_grouped_table(
            ot_labor_table,
            {
                "OT Units": "{:,.0f}",
                "Minimum Time Utilization": "{:.0%}",
                "Actual Time Utilization": "{:.1%}",
                "Available OT Hours per Worker": "{:,.2f}",
                "Registered OT Worker-Hours": "{:,.2f}",
                "Used OT Worker-Hours": "{:,.2f}",
                "Scheduled OT Worker-Hours": "{:,.2f}",
                "Line Workers": "{:,.0f}",
                "OT Labor Cost": "{:,.2f}",
            },
        )
    )
else:
    ot_labor_table = pd.DataFrame()
    print("No overtime labor was scheduled.")

if spare_ot_lines:
    print("DAY LINES WITH NO POSITIVE OT AVAILABILITY")
    display(pd.DataFrame({"Line": spare_ot_lines}).set_index("Line"))


## 13. OT WIP carryover


In [ ]:
ot_wip_rows = []

for day_position, day in enumerate(days):
    for product in products:
        route = product_routes[product]

        for current_process, next_process in zip(route, route[1:]):
            if day_position == 0:
                opening_units = 0.0
            else:
                opening_units = solution_value(
                    wip[
                        product,
                        current_process,
                        days[day_position - 1],
                    ]
                )

            closing_units = solution_value(
                wip[product, current_process, day]
            )
            if (
                opening_units <= SOLUTION_TOLERANCE
                and closing_units <= SOLUTION_TOLERANCE
            ):
                continue

            ot_wip_rows.append({
                "Date": pd.Timestamp(day_to_date[day]).date(),
                "Product": product,
                "Queue": f"{current_process} → {next_process}",
                "Opening OT WIP": opening_units,
                "Added in OT": solution_value(
                    stage_processed[product, current_process, day]
                ),
                "Moved Forward in OT": solution_value(
                    stage_processed[product, next_process, day]
                ),
                "Closing OT WIP": closing_units,
            })

if ot_wip_rows:
    ot_wip_carryover_table = pd.DataFrame(
        ot_wip_rows
    ).set_index(["Date", "Product", "Queue"])
    print("OVERTIME WIP CARRYOVER")
    display(ot_wip_carryover_table.style.format("{:,.0f}"))
else:
    ot_wip_carryover_table = pd.DataFrame()
    print("No overtime WIP carryover was scheduled.")

last_day = days[-1]
ending_ot_wip_snapshot = pd.DataFrame([
    {
        "As_Of_Date": pd.Timestamp(day_to_date[last_day]).date(),
        "Product_List": product,
        "Process": process,
        "OT_WIP_Units": round(
            solution_value(wip[product, process, last_day])
        ),
    }
    for product in products
    for process in product_routes[product][:-1]
])

print("ENDING OT-ONLY WIP SNAPSHOT")
display(
    ending_ot_wip_snapshot.set_index(
        ["As_Of_Date", "Product_List", "Process"]
    )
)
